In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def compute_hmf_with_uncertainties(
    true_masses, 
    predicted_mass_samples, 
    volume, 
    bins=20, 
    mass_range=(10, 13),
):

    mass_bins = np.linspace(mass_range[0], mass_range[1], bins+1)
    bin_centers = (mass_bins[1:] + mass_bins[:-1]) / 2
    bin_widths = mass_bins[1:] - mass_bins[:-1]
    
    counts, _ = np.histogram(true_masses, bins=mass_bins)
    true_hmf = counts / (volume * bin_widths)  
    #true_hmf_errs = np.sqrt(counts) / (volume * bin_widths)  # Poisson errors, might want to include in the future?
    
    n_samples = predicted_mass_samples.shape[0]
    sample_hmfs = np.zeros((n_samples, bins))
    
    for i in range(n_samples):
        counts, _ = np.histogram(predicted_mass_samples[i], bins=mass_bins)
        sample_hmfs[i] = counts / (volume * bin_widths)
    
    pred_hmf = np.median(sample_hmfs, axis=0)
    pred_hmf_lower = np.percentile(sample_hmfs, 16, axis=0)
    pred_hmf_upper = np.percentile(sample_hmfs, 84, axis=0)
    
    return bin_centers, true_hmf, pred_hmf, pred_hmf_lower, pred_hmf_upper

In [ ]:
def plot_hmf_comparison(bin_centers, true_hmf, 
                       pred_hmf, pred_hmf_lower, pred_hmf_upper):
    """Plot comparison between true and predicted HMFs with uncertainties"""
    plt.figure(figsize=(5, 3))
    
    
    plt.plot(bin_centers, pred_hmf, linestyle='-', color='indianred', label='Predicted HMF')
    plt.fill_between(bin_centers, pred_hmf_lower, pred_hmf_upper, 
                    color='indianred', alpha=0.3, label=r'$1\sigma$ confidence')
    
    plt.plot(bin_centers, true_hmf, color='black', label='True HMF', linewidth=2, linestyle='-')
    plt.yscale('log')
    plt.xlabel(r'log$_{10}$ (M$_{200c}$/M$_{\odot}$)')
    plt.ylabel(r'dn/dlog$_{10}$M [Mpc$^{-3}$]')
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    
    return plt.gca()


In [ ]:
train_sim = 'TNG' 
volume = boxsizes[train_sim]**3  
true_masses = truths[f'{train_sim}_{train_sim}']
predicted_mass_samples = predictions[f'{train_sim}_{train_sim}']
results = compute_hmf_with_uncertainties(true_masses, predicted_mass_samples, volume)
fig = plot_hmf_comparison(*results)
plt.savefig(f'paper_figures/hmf.pdf')